# 👂 UERC26 — Full Training + Evaluation Pipeline (Kaggle 2×T4)

This notebook runs on **Kaggle's 2× T4 GPUs** using DataParallel for ~2x training speedup.

**Pipeline:**
1. Train DINOv2 ViT-S/14 + AdaFace on public ear data
2. Generate embeddings for all splits + sequestered
3. Evaluate and compare with baseline

> **GPU Setup:** Settings (right panel) → Accelerator → **GPU T4 x2**

## Cell 1: Setup Kaggle Environment

In [ ]:
import os, shutil

INPUT_DIR = '/kaggle/input'
WORK_DIR = '/kaggle/working/uerc26'

# Deep search for train_pipeline.py (handles any nesting level)
print("Searching for project files...")
SOURCE = None
for root, dirs, files in os.walk(INPUT_DIR):
    if 'train_pipeline.py' in files:
        SOURCE = root
        print(f"Found project at: {SOURCE}")
        break
    if root.count(os.sep) - INPUT_DIR.count(os.sep) > 4:
        dirs.clear()

if SOURCE is None:
    print("\nERROR: Could not find train_pipeline.py!")
    print("\nAll files found under /kaggle/input/:")
    for root, dirs, files in os.walk(INPUT_DIR):
        for f in files[:10]:
            print(f"  {os.path.join(root, f)}")
        if root.count(os.sep) - INPUT_DIR.count(os.sep) > 3:
            dirs.clear()
else:
    print("Copying to working directory (Kaggle input is read-only)...")
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    shutil.copytree(SOURCE, WORK_DIR)
    os.chdir(WORK_DIR)
    print(f"Working directory: {os.getcwd()}")
    print(f"\nProject files:")
    for f in sorted(os.listdir('.')):
        if not f.startswith('.'):
            print(f"  {f}")

## Cell 2: Install Dependencies & Verify GPUs

In [ ]:
!pip install opencv-python-headless scipy -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

num_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {num_gpus}")

for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_mem / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")

if num_gpus >= 2:
    print(f"\n2x T4 GPUs detected! DataParallel will be used automatically.")
elif num_gpus == 1:
    print(f"\n1 GPU detected. Training will work fine, just no multi-GPU speedup.")
else:
    print(f"\nWARNING: No GPU! Enable GPU in Settings -> Accelerator -> GPU T4 x2")

import os
os.chdir(WORK_DIR)
data_path = 'UERC26_data/data/public'
assert os.path.exists(data_path), f"Data not found at {data_path}!"
num_subjects = len([d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))])
print(f"\nDataset OK: {num_subjects} subject folders found")

## RAM Check for Preloading

In [ ]:
import psutil
ram = psutil.virtual_memory()
print(f"RAM available: {ram.available/1e9:.1f} GB / {ram.total/1e9:.1f} GB total")
print(f"Dataset ~1.5 GB will be preloaded — need at least 3 GB free (2x for safety)")
assert ram.available > 3e9, "Not enough RAM for preloading! Disable --preload_images"
print("RAM check PASSED — preloading enabled")

## Cell 3: Quick Test — 5 Epochs (~10-15 min)

Run a quick test to verify everything works before the full training.

In [ ]:
os.chdir(WORK_DIR)

!python -u train_pipeline.py \
    --data_root UERC26_data/data/public \
    --backbone vits14 \
    --epochs 5 \
    --batch_size 256 \
    --eval_every 5 \
    --phase1_epochs 3 \
    --phase2_epochs 2 \
    --num_workers 4 \
    --preload_images \
    --save_dir checkpoints_test

print("\nQuick test passed! Proceed to full training below.")

## Cell 4: Full Training — 80 Epochs

**Expected time on 2×T4: ~1.5 hours**

The script automatically:
- Detects both GPUs and wraps model in DataParallel
- Uses Mixed Precision (AMP) for ~30-40% speedup
- LR warmup for first 3 epochs
- Phase 1 (ep 1-10): Frozen backbone, train head only
- Phase 2 (ep 11-50): Unfreeze last 4 transformer blocks
- Phase 3 (ep 51-80): Unfreeze everything
- Evaluates verification accuracy every 20 epochs (to save time)
- Saves best model automatically

> To use DINOv2 ViT-B (86M) instead, change `--backbone vits14` to `--backbone vitb14`
> and reduce `--batch_size` to 128

In [ ]:
os.chdir(WORK_DIR)

# ===== DINOv2 ViT-S/14 (21M params) — RECOMMENDED =====
!python -u train_pipeline.py \
    --data_root UERC26_data/data/public \
    --backbone vits14 \
    --epochs 80 \
    --batch_size 256 \
    --eval_every 20 \
    --num_workers 4 \
    --preload_images \
    --save_dir checkpoints

# ===== DINOv2 ViT-B/14 (86M params) — UNCOMMENT if ViT-S accuracy is low =====
# !python -u train_pipeline.py \
#     --data_root UERC26_data/data/public \
#     --backbone vitb14 \
#     --epochs 80 \
#     --batch_size 128 \
#     --eval_every 20 \
#     --num_workers 4 \
    --preload_images \
#     --save_dir checkpoints

## Cell 5: Generate Embeddings + Evaluate

Extracts embeddings for all splits + sequestered data with Test-Time Augmentation.

**~10-15 min on GPU**

In [ ]:
os.chdir(WORK_DIR)

!python -u generate_embeddings.py \
    --model_path checkpoints/best_model.pt \
    --public_root UERC26_data/data/public \
    --sequestered_root UERC26_data/data/sequestered_anonymized \
    --output_dir SUBMISSIONS/dinov2_solution \
    --batch_size 512 \
    --num_workers 4 \
    --preload_images \
    --use_tta

## Cell 6: Compare with Baseline — Results & Charts

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
os.chdir(WORK_DIR)

BASELINE = {'VER@0.1%': 0.2623, 'VER@1%': 0.4631, 'EER': 0.1569, 'AUC': 0.9215}

test_path = 'SUBMISSIONS/dinov2_solution/public_test_metrics.json'
if not os.path.exists(test_path):
    print(f"ERROR: {test_path} not found. Make sure Cell 5 completed successfully.")
else:
    with open(test_path) as f:
        m = json.load(f)
    
    print(f"{'='*70}")
    print(f"  YOUR DINOv2 + AdaFace  vs  BASELINE (ConvNeXt-Base)")
    print(f"{'='*70}")
    print(f"{'Metric':<15} {'Baseline':>12} {'Yours':>12} {'Diff':>10} {'Status':>10}")
    print(f"{'-'*70}")
    for k in ['VER@0.1%', 'VER@1%', 'EER', 'AUC']:
        base = BASELINE[k] * 100
        yours = m[k] * 100
        diff = yours - base
        if k == 'EER':
            status = 'BETTER' if diff < 0 else 'WORSE'
        else:
            status = 'BETTER' if diff > 0 else 'WORSE'
        icon = '\u2705' if status == 'BETTER' else '\u274c'
        print(f"{k:<15} {base:>11.2f}% {yours:>11.2f}% {diff:>+9.2f}% {icon} {status}")
    
    stats_path = 'SUBMISSIONS/dinov2_solution/model_stats.json'
    if os.path.exists(stats_path):
        with open(stats_path) as f:
            stats = json.load(f)
        print(f"\n{'='*70}")
        print(f"  MODEL SIZE COMPARISON")
        print(f"{'='*70}")
        print(f"  Baseline: 87.5M params, ~350 MB")
        print(f"  Yours:    {stats['num_parameters']/1e6:.1f}M params, ~{stats['model_size']/1e6:.0f} MB")
        print(f"  Reduction: {87.5 / (stats['num_parameters']/1e6):.1f}x fewer params")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    names = ['VER@0.1%', 'VER@1%', 'AUC']
    base_v = [BASELINE[k]*100 for k in names]
    your_v = [m[k]*100 for k in names]
    x = np.arange(len(names)); w = 0.35
    b1 = axes[0].bar(x-w/2, base_v, w, label='Baseline (ConvNeXt-B, 87.5M)', color='#e74c3c', alpha=0.85)
    b2 = axes[0].bar(x+w/2, your_v, w, label='Yours (DINOv2-S + AdaFace, 21M)', color='#2ecc71', alpha=0.85)
    axes[0].set_ylabel('Score (%)'); axes[0].set_title('Higher is Better \u2191', fontweight='bold', fontsize=13)
    axes[0].set_xticks(x); axes[0].set_xticklabels(names, fontsize=11)
    axes[0].legend(fontsize=9); axes[0].set_ylim(0, 100)
    for bar in list(b1)+list(b2):
        axes[0].annotate(f'{bar.get_height():.1f}%', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                         xytext=(0,5), textcoords='offset points', ha='center', fontsize=9)
    b3 = axes[1].bar(['Baseline\n(87.5M)', 'Yours\n(21M)'], [BASELINE['EER']*100, m['EER']*100],
                     color=['#e74c3c', '#2ecc71'], alpha=0.85, width=0.5)
    axes[1].set_ylabel('EER (%)'); axes[1].set_title('Lower is Better \u2193', fontweight='bold', fontsize=13)
    for bar in b3:
        axes[1].annotate(f'{bar.get_height():.1f}%', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                         xytext=(0,5), textcoords='offset points', ha='center', fontsize=11)
    plt.suptitle('UERC26: Your DINOv2 + AdaFace vs Baseline', fontsize=15, fontweight='bold')
    plt.tight_layout(); plt.savefig('comparison_chart.png', dpi=150, bbox_inches='tight'); plt.show()

## Cell 7: Save & Download Results

In [ ]:
import shutil
os.chdir(WORK_DIR)

output_zip = '/kaggle/working/uerc26_output'
os.makedirs(output_zip, exist_ok=True)

if os.path.exists('checkpoints'):
    shutil.copytree('checkpoints', os.path.join(output_zip, 'checkpoints'), dirs_exist_ok=True)
if os.path.exists('SUBMISSIONS/dinov2_solution'):
    shutil.copytree('SUBMISSIONS/dinov2_solution', os.path.join(output_zip, 'SUBMISSIONS/dinov2_solution'), dirs_exist_ok=True)
if os.path.exists('comparison_chart.png'):
    shutil.copy2('comparison_chart.png', output_zip)

shutil.make_archive('/kaggle/working/uerc26_final_results', 'zip', output_zip)

print("RESULTS SAVED!")
print(f"\nDownload from: Output tab (right panel) -> uerc26_final_results.zip")
print(f"\nFiles included:")
total_size = 0
for root, dirs, flist in os.walk(output_zip):
    for f in flist:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath) / 1e6
        total_size += size
        rel = os.path.relpath(fpath, output_zip)
        print(f"  {rel:<50s} ({size:.1f} MB)")
print(f"\nTotal: {total_size:.0f} MB")